# Εργαστήριο 1 — Αναπαράσταση Προβλήματος και Αναζήτηση χωρίς Πληροφόρηση

> ### Γενικές οδηγίες υποβολής
> - Συμπληρώστε το `STUDENT_ID` με τον Αριθμό Μητρώου σας. Από αυτόν δημιουργείται το προσωπικό σας instance.
> - Μην αλλάζετε τα ονόματα των συναρτήσεων που ζητούνται.
> - Πριν την υποβολή: **Kernel → Restart & Run All**. Το Notebook πρέπει να εκτελείται χωρίς σφάλμα από την αρχή έως το τέλος.
> - Οι σύντομες απαντήσεις πρέπει να βασίζονται στο **δικό σας αποτέλεσμα και στα δικά σας βήματα εκτέλεσης**.

In [1]:
STUDENT_ID = "3323"

import hashlib, random
from collections import deque

def make_graph(student_id):
    """Δημιουργεί προσωπικό κατευθυνόμενο γράφο με εγγυημένη λύση."""
    seed = int(hashlib.sha256(("LAB1-" + student_id).encode()).hexdigest()[:8], 16)
    rng = random.Random(seed)

    # List of Nodes & creation of dictionary of aln nodes: unpopulated A : [], B : []...
    nodes = ["S","A","B","C","D","E","F","H","G"]
    graph = {n: [] for n in nodes}

    # Εγγυημένος κορμός λύσης
    base_edges = [("S","A"),("A","D"),("D","G")]
    # Populates nodes S-> `A`, A-> `D`, D-> `G`:
    for u,v in base_edges:
        graph[u].append(v)

    # Πρόσθετες ακμές, ορισμένες δημιουργούν εναλλακτικά μονοπάτια
    candidates = [
        ("S","B"),("S","C"),("A","B"),("A","E"),("B","D"),
        ("B","F"),("C","B"),("C","E"),("C","H"),("E","D"),
        ("E","H"),("F","E"),("F","G"),("H","F"),("H","G")
    ]

    # Shuffles and appends the 10 first to the graph
    rng.shuffle(candidates)
    for u,v in candidates[:10]:
        graph[u].append(v) 

    # Sorts and creates a set to remove duplicates
    for u in graph:
        graph[u] = sorted(set(graph[u])) 
    return graph, "S", "G"

graph, start, goal = make_graph(STUDENT_ID)
print("Αρχική κατάσταση:", start)
print("Τελική κατάσταση:", goal)
print("Γράφος:", graph)
for n in graph:
    print(f"  {n} -> {graph[n]}")

Αρχική κατάσταση: S
Τελική κατάσταση: G
Γράφος: {'S': ['A', 'B', 'C'], 'A': ['B', 'D'], 'B': ['D', 'F'], 'C': ['B'], 'D': ['G'], 'E': ['D', 'H'], 'F': ['E', 'G'], 'H': [], 'G': []}
  S -> ['A', 'B', 'C']
  A -> ['B', 'D']
  B -> ['D', 'F']
  C -> ['B']
  D -> ['G']
  E -> ['D', 'H']
  F -> ['E', 'G']
  H -> []
  G -> []


In [2]:

def _search(graph, start, goal, strategy="bfs", limit=None):

    frontier = deque([[start]])
    print(frontier)
    expanded = []
    trace = []

    while frontier:
        frontier_snapshot = [list(p) for p in frontier]
        path = frontier.popleft()
        current = path[-1]
        expanded.append(current)

        if current == goal:
            trace.append({
                "frontier": frontier_snapshot,
                "current": current,
                "new_paths": [],
            })
            return {"path": path, "expanded": expanded, "trace": trace}

        new_paths = [] 
        if limit is None or len(path) - 1 < limit:
            for neighbor in graph.get(current, []):
                if neighbor not in path:
                    new_paths.append(path + [neighbor])

        trace.append({
            "frontier": frontier_snapshot,
            "current": current,
            "new_paths": new_paths,
        })

        if strategy == "bfs":
            frontier.extend(new_paths)
        elif strategy == "dfs":
            frontier.extendleft(reversed(new_paths)) # preserve left to right order

    return {"path": None, "expanded": expanded, "trace": trace}



## Μέρος Β — Αναζήτηση Πρώτα κατά Βάθος (DFS) (25%)

Υλοποιήστε `dfs(graph, start, goal)`.

Η συνάρτηση πρέπει να επιστρέφει λεξικό με:
- `path`: **διαδρομή λύσης**,
- `expanded`: **Σειρά επέκτασης κόμβων**,
- `trace`: **καταγραφή των βημάτων εκτέλεσης**.

Κάθε εγγραφή του `trace` να περιέχει:
- `frontier`: το **Σύνορο πριν από την επέκταση**,
- `current`: τον **κόμβο που θα επεκταθεί**,
- `new_paths`: τις **νέες διαδρομές** που προστέθηκαν.

Οι διάδοχοι επεκτείνονται αλφαβητικά. Ο έλεγχος στόχου γίνεται όταν ο κόμβος αφαιρείται από το Σύνορο.

In [3]:
def dfs(graph, start, goal):
    return _search(graph, start, goal, strategy="dfs")

print(dfs(graph, start, goal)["path"])

deque([['S']])
['S', 'A', 'B', 'D', 'G']


## Μέρος Γ — Αναζήτηση Πρώτα κατά Πλάτος (BFS) (25%)

Υλοποιήστε `bfs(graph, start, goal)` με το ίδιο format αποτελέσματος.

In [4]:
def bfs(graph, start, goal):
    return _search(graph, start, goal, strategy="bfs")

print(bfs(graph, start, goal)["path"])

deque([['S']])
['S', 'A', 'D', 'G']


## Μέρος Δ — Επαναληπτική Εμβάθυνση (IDS) (25%)

Υλοποιήστε:
- `depth_limited_search(graph, start, goal, limit)` — Αναζήτηση Περιορισμένου Βάθους,
- `ids(graph, start, goal, max_depth=20)` — Αλγόριθμος Επαναληπτικής Εμβάθυνσης.

Το `ids` πρέπει να επιστρέφει:
- `path`: διαδρομή λύσης,
- `found_depth`: το **όριο εμβάθυνσης στο οποίο βρέθηκε η λύση**,
- `iterations`: για κάθε όριο εμβάθυνσης, τη σειρά κόμβων που εξερευνήθηκαν.

In [5]:
def depth_limited_search(graph, start, goal, limit):
     return _search(graph, start, goal, strategy="dfs", limit=limit)

def ids(graph, start, goal, max_depth=20):
    iterations = {}
    for depth in range(max_depth + 1):
        res = depth_limited_search(graph, start, goal, depth)
        iterations[depth] = res["expanded"]
        if res["path"] is not None:
            return {
                "path": res["path"],
                "found_depth": depth,
                "iterations": iterations,
            }
    return {"path": None, "found_depth": None, "iterations": iterations}
print(ids(graph, start, goal)["path"])

deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])
['S', 'A', 'D', 'G']


## Μέρος Ε — Εκτέλεση, πίνακες και ερμηνεία (10%)

1. Εκτελέστε DFS, BFS και IDS.
2. Για DFS και BFS εμφανίστε πίνακα με στήλες:

**Βήμα | Σύνορο | Κόμβος που θα επεκταθεί | Νέες διαδρομές**

3. Απαντήστε:
   - DFS και BFS επέστρεψαν ίδια διαδρομή λύσης;
   - Ποιος επέκτεινε λιγότερους κόμβους στο προσωπικό σας instance;
   - Σε ποιο όριο εμβάθυνσης βρήκε λύση ο IDS;

In [13]:
import pandas as pd

def trace_rows(result):
    return [
        {
            "Βήμα": i,
            "Σύνορο": "; ".join(" => ".join(p) for p in step["frontier"]),
            "Κόμβος": step["current"],
            "Νέες διαδρομές": "; ".join(" => ".join(p) for p in step["new_paths"]),
        }
        for i, step in enumerate(result["trace"], 1)
    ]
dfs_result = dfs(graph, start, goal)
bfs_result = bfs(graph, start, goal)
ids_result = ids(graph, start, goal)
dfs_df, bfs_df = pd.DataFrame(trace_rows(dfs_result)), pd.DataFrame(trace_rows(bfs_result))
# for df in (dfs_df, bfs_df):
#     print(len(df), df["Κόμβος"].nunique())
display(dfs_df, bfs_df)

deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])


,Βήμα,Σύνορο,Κόμβος,Νέες διαδρομές
0,1,S,S,S => A; S => B; S => C
1,2,S => A; S => B; S => C,A,S => A => B; S => A => D
2,3,S => A => B; S => A => D; S => B; S => C,B,S => A => B => D; S => A => B => F
3,4,S => A => B => D; S => A => B => F; S => A => ...,D,S => A => B => D => G
4,5,S => A => B => D => G; S => A => B => F; S => ...,G,


,Βήμα,Σύνορο,Κόμβος,Νέες διαδρομές
0,1,S,S,S => A; S => B; S => C
1,2,S => A; S => B; S => C,A,S => A => B; S => A => D
2,3,S => B; S => C; S => A => B; S => A => D,B,S => B => D; S => B => F
3,4,S => C; S => A => B; S => A => D; S => B => D;...,C,S => C => B
4,5,S => A => B; S => A => D; S => B => D; S => B ...,B,S => A => B => D; S => A => B => F
5,6,S => A => D; S => B => D; S => B => F; S => C ...,D,S => A => D => G
6,7,S => B => D; S => B => F; S => C => B; S => A ...,D,S => B => D => G
7,8,S => B => F; S => C => B; S => A => B => D; S ...,F,S => B => F => E; S => B => F => G
8,9,S => C => B; S => A => B => D; S => A => B => ...,B,S => C => B => D; S => C => B => F
9,10,S => A => B => D; S => A => B => F; S => A => ...,D,S => A => B => D => G


In [ ]:
import pandas as pd


def formatPaths(paths):
   
    return "; ".join(" => ".join(p) for p in paths)


def buildTraceDf(result):
    
    rows = [
        {
            "Βήμα": i,
            "Σύνορο": formatPaths(step["frontier"]),
            "Κόμβος": step["current"],
            "Νέες διαδρομές": formatPaths(step["new_paths"]),
        }
        for i, step in enumerate(result["trace"], 1)
    ]
    return pd.DataFrame(rows)


def buildIdsDf(ids_result):

    rows = [
        {
            "Όριο Βάθους": depth,
            "Σειρά Επέκτασης": " -> ".join(expanded),
            "Επεκτάσεις": len(expanded),
        }
        for depth, expanded in ids_result["iterations"].items()
    ]
    return pd.DataFrame(rows)


search_results = {
    "DFS": dfs(graph, start, goal),
    "BFS": bfs(graph, start, goal),
}
ids_result = ids(graph, start, goal)

# Disable column width truncation so full frontier paths remain visible.
with pd.option_context("display.max_colwidth", None):
    for name, res in search_results.items():
        print(f"=== {name} ===")
        print(f"Διαδρομή: {' => '.join(res['path'])} | Επεκτάσεις: {len(res['expanded'])}")
        display(buildTraceDf(res))

    print("=== IDS ===")
    print(
        f"Διαδρομή: {' => '.join(ids_result['path'])} "
        f"| Βάθος λύσης: {ids_result['found_depth']} "
        f"| Συνολικές επεκτάσεις: {sum(len(v) for v in ids_result['iterations'].values())}"
    )
    display(buildIdsDf(ids_result))

deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])
deque([['S']])
=== DFS ===
Διαδρομή: S => A => B => D => G | Επεκτάσεις: 5


,Βήμα,Σύνορο,Κόμβος,Νέες διαδρομές
0,1,S,S,S => A; S => B; S => C
1,2,S => A; S => B; S => C,A,S => A => B; S => A => D
2,3,S => A => B; S => A => D; S => B; S => C,B,S => A => B => D; S => A => B => F
3,4,S => A => B => D; S => A => B => F; S => A => D; S => B; S => C,D,S => A => B => D => G
4,5,S => A => B => D => G; S => A => B => F; S => A => D; S => B; S => C,G,


=== BFS ===
Διαδρομή: S => A => D => G | Επεκτάσεις: 12


,Βήμα,Σύνορο,Κόμβος,Νέες διαδρομές
0,1,S,S,S => A; S => B; S => C
1,2,S => A; S => B; S => C,A,S => A => B; S => A => D
2,3,S => B; S => C; S => A => B; S => A => D,B,S => B => D; S => B => F
3,4,S => C; S => A => B; S => A => D; S => B => D; S => B => F,C,S => C => B
4,5,S => A => B; S => A => D; S => B => D; S => B => F; S => C => B,B,S => A => B => D; S => A => B => F
5,6,S => A => D; S => B => D; S => B => F; S => C => B; S => A => B => D; S => A => B => F,D,S => A => D => G
6,7,S => B => D; S => B => F; S => C => B; S => A => B => D; S => A => B => F; S => A => D => G,D,S => B => D => G
7,8,S => B => F; S => C => B; S => A => B => D; S => A => B => F; S => A => D => G; S => B => D => G,F,S => B => F => E; S => B => F => G
8,9,S => C => B; S => A => B => D; S => A => B => F; S => A => D => G; S => B => D => G; S => B => F => E; S => B => F => G,B,S => C => B => D; S => C => B => F
9,10,S => A => B => D; S => A => B => F; S => A => D => G; S => B => D => G; S => B => F => E; S => B => F => G; S => C => B => D; S => C => B => F,D,S => A => B => D => G


=== IDS ===
Διαδρομή: S => A => D => G | Βάθος λύσης: 3 | Συνολικές επεκτάσεις: 21


,Όριο Βάθους,Σειρά Επέκτασης,Επεκτάσεις
0,0,S,1
1,1,S -> A -> B -> C,4
2,2,S -> A -> B -> D -> B -> D -> F -> C -> B,9
3,3,S -> A -> B -> D -> F -> D -> G,7
